# GCM BA - 7b. Ad hoc test: MBCn and the snow, evaluated on the first 34 years of the runs

**Why 7b.** In `7_test_mbcn` the changes of the MBCn version were impossible (warming turned into a cooling of 2 °C,
pr −27 %, dry days +20 points). Cause: `xsdba` 0.6.1 `MBCn.adjust` (`mbcn_adjust`) builds the time groups (5 days ×
19) from the time axis of `ref` (34 years, 12 410 days) and uses them to index `sim`: for the 50-year runs (18 250
days) only the first 12 410 days are adjusted, the last 16 years stay at zero (`xr.zeros_like(sim)`). The historical
product and the first 34 years of each run are correct. 7b does not rerun MBCn: it re-evaluates the products of 7 on
the first 34 years of every run, for every version (raw, F1, ranked jitter, MBCn), so the comparison is fair. For a
full product, `sim` must be adjusted in blocks of the length of `ref` (e.g. `xsdba.stack_periods` and `period_dim`).

**Ad hoc test, outside the F1 chain**: it reads the inputs of notebook 2, the F1 products of notebook 3, the products
of `6_test_jitter`, the products and caches of `7_test_mbcn` and the statistics cached by notebook 5, and writes only
in `GCM_BA/7b_test_mbcn_first34y/` (EOS) and `outputs/` next to it.

The description of 7 follows.

**Question.** F1, a univariate quantile delta mapping per pixel and variable, keeps the dependence of EC-Earth3
between precipitation and temperature, including its bias against ERA5: in winter the daily pr – tasmax correlation
is too strong (Spearman DJF: ERA5 0.216, raw 0.257, F1 0.260). For LISFLOOD this dependence drives the snow: the
precipitation of a day falls as snow when the mean daily temperature is below `TempSnow` (1 °C in the LISFLOOD
documentation), so the snow accumulation, and the mass balance of the glaciers, depends on how much precipitation
falls on the cold days, not only on the totals. A multivariate correction, MBCn (Cannon 2018, *Clim. Dyn.* 50,
31–49), imposes the dependence of ERA5 between the variables. Does it improve the snow accumulation, and does it keep
the change simulated under the AMOC weakening (the cooling should raise the share of the precipitation falling as
snow)?

**MBCn (xsdba).** `xsdba.MBCn`: N-dimensional pdf transform (random rotations of the standardised variables and
quantile mapping, `N_ITER` iterations, Pitié et al. 2005) trained on ERA5 vs EC-Earth3 historical 1981–2014, then a
univariate QDM of each variable (pr multiplicative, tasmax additive, daily range multiplicative, as F1) reordered
along the ranks of the multivariate transform. Windows of 5 days × 19 (95 days, close to the 3 months of F1;
`time.month` is not supported by MBCn). pr: ranked jitter of `6_test_jitter` before, values < 0.1 mm/day set to 0
after; tasmin = tasmax − daily range. Applied to the domain pixels only (European land of LISFLOOD, where every
statistic is computed).

**Compared**: ERA5, raw EC-Earth3, F1, F1 with the ranked jitter (pr of `6_test_jitter`, temperatures of F1) and MBCn.
- **Snow** (cold season, November–April): snowfall fraction = precipitation of the days with mean temperature
  (tasmax + tasmin) / 2 < `TEMP_SNOW` / total precipitation; snow accumulation (mm per year); positive degree-days
  above `TEMP_MELT` (melt, per year); over the domain and the mountains (Alps, Scandinavia).
- **Calibration** against ERA5 (means, dry days, Rx1day) and the **dependence** statistics of 5_dependence.
- **Changes** (warming: original − historical; AMOC weakening: hosing − original): domain and regional means and the
  pattern regression of the adjusted change on the raw change (slope 1 = kept).

**Decision.** MBCn is worth adding to the RCM correction if it brings the snow accumulation clearly closer to ERA5
than F1 and keeps the change of the snowfall fraction and of the dependence with the hosing; otherwise the
univariate correction (with the ranked jitter) is kept and the dependence bias of the model is a stated limit.

**Limits.** The 0.7° grid resolves the Alps with a few cells: this is a bench for the RCM correction (0.25°), where the
test should be repeated, and LISFLOOD corrects the temperature for the elevation of its own grid.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, time, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
import validation_utils as vu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
CAL_PERIOD = ("1981", "2014")      # as F1
REGION = bu.gcm_ba_region(cfg, 1.0)
F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}_{bu.F1_VARIANT}" + ("_europe" if REGION else "")
RANK_TAG = f"{F1_TAG}_rankjit"                      # products of 6_test_jitter
TEST_TAG = f"{F1_TAG}_mbcn"
GCM_RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
RUNS = ["gcm_historical"] + GCM_RUNS
TEMP_SNOW = 1.0                    # degC, mean daily temperature below which pr falls as snow (LISFLOOD TempSnow)
TEMP_MELT = 1.0                    # degC, threshold of the positive degree-days (LISFLOOD TempMelt, 0 or 1 in its docs)
COLD = [11, 12, 1, 2, 3, 4]        # cold season of the snow statistics
N_ITER = 20                        # MBCn iterations (Cannon 2018)
NQ = 50                            # quantiles of the npdf transform and of the final QDM
WINDOW = 19                        # windows of 5 days x 19 = 95 days (about the 3 months of F1)
SEED = 20261109
LOC_CHUNK = 400                    # domain pixels per dask block
OVERWRITE = False
MOUNTAINS = {"Alps": {"lat": (43.5, 48.5), "lon": (5.0, 16.5)}, "Scandinavia": {"lat": (58.0, 71.0), "lon": (4.0, 20.0)}}

DOMAIN = vu.init(cfg, "domain", borders_res="01M")        # LISFLOOD domain without the land outside Europe (as 5)
OUTF1 = bu.gcm_ba_dir(cfg, "GCM_bias_adjusted_0.7deg")    # F1 products (notebook 3)
RANK = bu.gcm_ba_dir(cfg, "6_test_jitter", RANK_TAG)       # products and cache of 6_test_jitter (read only)
CACHE5 = bu.gcm_ba_dir(cfg, "5_dependence", "cache" + ("_europe" if REGION else ""))   # statistics of 5 (read only)
TEST7 = bu.gcm_ba_dir(cfg, "7_test_mbcn", TEST_TAG)       # 7: products and caches (read only)
PDIR, CACHE7 = TEST7 / "products", TEST7 / "cache"
TEST = bu.gcm_ba_dir(cfg, "7b_test_mbcn_first34y", TEST_TAG)   # the only folder written on EOS
CACHE = TEST / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
LABEL = {"gcm_historical": "historical", "gcm_2025_nohosing": "2025 original", "gcm_2025_hosing": "2025 hosing",
         "gcm_2050_nohosing": "2050 original", "gcm_2050_hosing": "2050 hosing"}
PAIRS = [("gcm_2025_nohosing", "gcm_historical", "2025 original - historical"),
         ("gcm_2050_nohosing", "gcm_historical", "2050 original - historical"),
         ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"),
         ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original")]
KINDS = ["raw", "F1", "rank", "mbcn"]
period_of = lambda name: CAL_PERIOD if name in ("era5", "gcm_historical") else None
print(f"F1 {F1_TAG} | ranked jitter {RANK_TAG} | MBCn {N_ITER} iterations, {NQ} quantiles, 5 d x {WINDOW} windows | "
      f"snow below {TEMP_SNOW} degC | written in {TEST}")

## 1. Check of the diagnosis: length of the products and share of zeros in their last days

N_KEEP = number of days of the historical product (34 years). In every run, the days after N_KEEP should be all zero
(never adjusted) and the first N_KEEP days should look like the historical product.

In [ ]:
with xr.open_dataset(PDIR / f"gcm_historical_{TEST_TAG}.nc") as m:
    N_KEEP = m.sizes["time"]
rows = []
for name in RUNS:
    with xr.open_dataset(PDIR / f"{name}_{TEST_TAG}.nc") as m:
        tx = m["tasmax"].values
        rows.append({"run": name, "days": tx.shape[0], "first day": str(m.time.values[0])[:10],
                     "zeros, first N_KEEP days (%)": 100 * float((tx[:N_KEEP] == 0).mean()),
                     "zeros, after N_KEEP (%)": 100 * float((tx[N_KEEP:] == 0).mean()) if tx.shape[0] > N_KEEP else np.nan,
                     "mean tasmax, first N_KEEP days": float(tx[:N_KEEP].mean())})
print(f"N_KEEP = {N_KEEP} days")
display(pd.DataFrame(rows).set_index("run").round(2))

## 2. Snow, calibration and dependence of every version

Per dataset and version, per pixel: snowfall fraction and accumulation of the cold season (November–April), positive
degree-days, mean pr, dry days, Rx1day, mean tasmax / tasmin; and the dependence statistics of 5_dependence (ERA5, raw
and F1 from the cache of notebook 5, the ranked jitter from the cache of 6_test_jitter, computed here if missing).

In [ ]:
def version(name, kind):
    # (pr, tasmax, tasmin) of a dataset on the domain pixels, as vu.domain_pixels dicts
    per = period_of(name)
    cut = lambda da: da if name in ("era5", "gcm_historical") else da.isel(time=slice(0, N_KEEP))   # 7b: first 34 years
    sub = lambda da: cut(da if per is None else bu.subset_time(da, *per))
    if kind == "raw":
        return [vu.domain_pixels(cut(bu.gcm_ba_input(cfg, name, v, per, REGION))) for v in ("pr", "tasmax", "tasmin")]
    f1 = xr.open_dataset(OUTF1 / f"{name}_{F1_TAG}.nc", chunks={"time": 3100}, **bu._TIME_KW)
    if kind in ("F1", "rank"):
        pr = f1["pr"] if kind == "F1" else xr.open_dataset(RANK / "products" / f"{name}_{RANK_TAG}.nc", chunks={"time": 3100}, **bu._TIME_KW)["pr"]
        return [vu.domain_pixels(sub(pr)), vu.domain_pixels(sub(f1["tasmax"])), vu.domain_pixels(sub(f1["tasmin"]))]
    m = xr.open_dataset(PDIR / f"{name}_{TEST_TAG}.nc")                       # MBCn: already on the domain pixels
    ref = vu.domain_pixels(sub(f1["pr"]))
    n = ref["x"].shape[0]
    return [{**ref, "x": m[v].values[:n].astype("float32")} for v in ("pr", "tasmax", "tasmin")]


def snow_stats(pr, tx, tn):
    x, t = pr["x"], (tx["x"] + tn["x"]) / 2
    mon, yr = pr["month"], pr["year"]
    ny = np.unique(yr).size
    cold = np.isin(mon, COLD)
    sn = np.where(cold[:, None] & (t < TEMP_SNOW), x, 0.0)
    tot = np.where(cold[:, None], x, 0.0)
    with np.errstate(invalid="ignore", divide="ignore"):
        frac = sn.sum(0) / tot.sum(0)
    rx = np.stack([np.nanmax(x[yr == y], axis=0) for y in np.unique(yr)])
    return pd.DataFrame({"snowfall fraction, Nov-Apr": frac, "snow accumulation (mm/year)": sn.sum(0) / ny,
                         "positive degree-days (degC day/year)": np.maximum(t - TEMP_MELT, 0).sum(0) / ny,
                         "mean pr (mm/day)": np.nanmean(x, 0), "dry days (< 0.1 mm, %)": 100 * np.nanmean(x < bu.THRESH_PR, 0),
                         "Rx1day (mm/day)": np.nanmean(rx, 0), "mean tasmax (degC)": np.nanmean(tx["x"], 0),
                         "mean tasmin (degC)": np.nanmean(tn["x"], 0)})


SNOW, DEP = {}, {}
for name, kinds in [("era5", ["raw"])] + [(n, KINDS) for n in RUNS]:
    for k in kinds:
        key = "ERA5" if name == "era5" else f"{LABEL[name]} {k}"
        full = name in ("era5", "gcm_historical")          # 7b: whole period, the caches of 7 are valid
        ps = (CACHE7 if full and (CACHE7 / f"snow_{name}_{k}.pkl").exists() else CACHE) / f"snow_{name}_{k}.pkl"
        tag = {"raw": "raw", "F1": F1_TAG, "rank": RANK_TAG, "mbcn": TEST_TAG}[k]
        if full:
            pd_ = {"raw": CACHE5, "F1": CACHE5, "rank": RANK / "cache", "mbcn": CACHE7}[k] / f"{name}_{tag}_dependence.pkl"
        else:
            pd_ = CACHE / f"{name}_{tag}_first34y_dependence.pkl"
        need_dep = not pd_.exists() and not (CACHE / pd_.name).exists()
        if ps.exists() and not OVERWRITE and not need_dep:
            SNOW[key] = pd.read_pickle(ps)
        else:
            pr, tx, tn = version(name, k)
            SNOW[key] = snow_stats(pr, tx, tn); SNOW[key].to_pickle(CACHE / ps.name)
            if need_dep:
                vu.dependence_stats(pr, tx, CACHE / pd_.name, True, label=key)
            del pr, tx, tn
        DEP[key] = pd.read_pickle(pd_ if pd_.exists() else CACHE / pd_.name)
LAT, LON = DEP["ERA5"]["lat"], np.mod(DEP["ERA5"]["lon"] + 180, 360) - 180
W = np.cos(np.deg2rad(LAT))
ZONES = {"domain": np.ones(LAT.size, bool),
         **{k: (LAT >= b["lat"][0]) & (LAT <= b["lat"][1]) & (LON >= b["lon"][0]) & (LON <= b["lon"][1]) for k, b in MOUNTAINS.items()}}
wmean = lambda a, z: float(np.nansum((W * a)[z]) / np.sum((W * np.isfinite(a))[z]))
print({k: int(z.sum()) for k, z in ZONES.items()}, "pixels")

## 3. Historical climate 1981–2014: snow, calibration and dependence against ERA5

In [ ]:
rows = []
for c in SNOW["ERA5"].columns:
    for zn, z in ZONES.items():
        if zn != "domain" and not c.startswith(("snow", "positive")):
            continue
        rows.append({"statistic": c, "zone": zn, "ERA5": wmean(SNOW["ERA5"][c].values, z),
                     **{k: wmean(SNOW[f"historical {k}"][c].values, z) for k in KINDS}})
HIST = pd.DataFrame(rows).set_index(["statistic", "zone"])
HIST.to_csv(OUT / f"snow_calibration_historical_{TEST_TAG}.csv")
rows = []
E = DEP["ERA5"]["temporal"]
for c in E.columns:
    rows.append({"statistic": c, "ERA5": wmean(E[c].values, ZONES["domain"]),
                 **{k: wmean(DEP[f"historical {k}"]["temporal"][c].values, ZONES["domain"]) for k in KINDS}})
TEMP = pd.DataFrame(rows).set_index("statistic")
TEMP.to_csv(OUT / f"dependence_historical_{TEST_TAG}.csv")
with pd.option_context("display.width", 250, "display.max_rows", 200):
    print("(1) snow and calibration, historical (area-weighted means)"); display(HIST.round(3))
    print("(2) dependence statistics, historical, domain"); display(TEMP.round(3))

## 4. Changes: is the change of the snow and of the dependence kept?

For each pair (runs: first 34 years), the area-weighted change of each statistic (snowfall fraction in points;
accumulation, degree-days, mean pr and Rx1day in % of the zone mean; dependence statistics in their units) for raw and
the four versions, and the pattern regression of the adjusted change on the raw change over the domain, per pixel in
the units of the statistic (slope 1 = kept).

In [ ]:
def wslope(x, y, z):
    ok = np.isfinite(x) & np.isfinite(y) & z
    w = W[ok]; x, y = x[ok], y[ok]
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    vx = np.average((x - mx) ** 2, weights=w)
    return np.average((x - mx) * (y - my), weights=w) / vx if vx > 0 else np.nan


REL = ("snow accumulation (mm/year)", "positive degree-days (degC day/year)", "mean pr (mm/day)", "Rx1day (mm/day)")
rows = []
for a_, b_, lab in PAIRS:
    ka, kb = LABEL[a_], LABEL[b_]
    for c in SNOW["ERA5"].columns:
        for zn, z in ZONES.items():
            if zn != "domain" and not c.startswith(("snow", "positive")):
                continue
            d, chg = {}, {}
            for k in ["raw"] + KINDS[1:]:
                x1, x0 = SNOW[f"{ka} {k}"][c].values, SNOW[f"{kb} {k}"][c].values
                d[k] = 100 * (x1 - x0) if c.startswith("snowfall") else x1 - x0          # per pixel, for the slopes
                # 7b: relative changes on the zone means (no infinite percentages where a pixel has no snow)
                chg[k] = 100 * (wmean(x1, z) / wmean(x0, z) - 1) if c in REL else wmean(d[k], z)
            rows.append({"pair": lab, "statistic": c + (" (%)" if c in REL else " (points)" if c.startswith("snowfall") else ""),
                         "zone": zn, **{f"change {k}": v for k, v in chg.items()},
                         **{f"slope {k}": wslope(d["raw"], d[k], z) for k in KINDS[1:]}})
    for c in E.columns:
        d = {k: DEP[f"{ka} {k}"]["temporal"][c].values - DEP[f"{kb} {k}"]["temporal"][c].values for k in KINDS}
        rows.append({"pair": lab, "statistic": c, "zone": "domain", **{f"change {k}": wmean(v, ZONES["domain"]) for k, v in d.items()},
                     **{f"slope {k}": wslope(d["raw"], d[k], ZONES["domain"]) for k in KINDS[1:]}})
CHG = pd.DataFrame(rows).set_index(["pair", "statistic", "zone"])
CHG.to_csv(OUT / f"changes_{TEST_TAG}.csv")
key = ("snowfall", "snow accumulation", "positive degree", "Spearman", "P(wet | wet)", "mean dry spell", "CDD")
with pd.option_context("display.width", 250, "display.max_rows", 300):
    display(CHG[CHG.index.get_level_values("statistic").str.startswith(key)].round(3))

## 5. Reading

- **Snow, historical** (section 3): snowfall fraction and accumulation of F1, ranked jitter and MBCn against ERA5,
  over the domain and the mountains; MBCn should also bring the winter pr – tasmax Spearman towards ERA5 (0.216).
- **Calibration**: MBCn must stay as close to ERA5 as F1 on the means, dry days and Rx1day (its last step is a QDM).
- **Changes** (section 4): the change of the snowfall fraction and of the accumulation with the hosing (expected
  positive: cooling), and the change of the dependence statistics, raw against each version (slope near 1 = kept).
  MBCn is worth it for the RCM correction only if the gain on the historical snow is clear and these changes are
  kept.